In [1]:
# Cell 1: Cài đặt thư viện cần thiết để trích xuất cache landmark (InsightFace CUDA, ONNX Runtime GPU) trên Kaggle / Google Colab
import os, sys, platform, subprocess

if platform.system() == "Linux":
    # Cài đặt các gói hệ thống cần thiết cho OpenCV và ONNX Runtime
    print("Đang cấu hình gói hệ thống trên Linux...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True
    )

# Cài đặt các thư viện Python: InsightFace, ONNX Runtime GPU, OpenCV, v.v.
print("Đang cài đặt các thư viện Python (InsightFace CUDA, ONNX Runtime GPU, OpenCV)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
    "insightface", "onnxruntime-gpu==1.21.0", "opencv-python", "tqdm", "pyyaml"
])
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")

Đang cấu hình gói hệ thống trên Linux...


Đang cài đặt các thư viện Python (InsightFace CUDA, ONNX Runtime GPU, OpenCV)...
Cài đặt hoàn tất! Môi trường đã sẵn sàng.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nyvantran6634/data-processed-v3", output_dir="./kaggle")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'data-processed-v3' dataset.
Path to dataset files: /kaggle/input/data-processed-v3


In [3]:
# Cell 2: Import các thư viện cần thiết và cấu hình môi trường ONNX Runtime CUDA
from __future__ import annotations
import csv
import json
import logging
import math
import os
import random
import sys
import tempfile
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
if hasattr(sys.stderr, "reconfigure"):
    try:
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Cấu hình nạp DLL PyTorch CUDA và môi trường cho ONNX Runtime trên Windows / Linux
torch_lib_path = os.path.join(os.path.dirname(torch.__file__), "lib")
if os.path.exists(torch_lib_path):
    try:
        os.add_dll_directory(torch_lib_path)
    except Exception:
        pass
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

logger = logging.getLogger("DriverGuardian.Cache")

In [4]:
# Cell 3: Hệ thống cấu hình trung tâm (config.py)
@dataclass
class TrainConfig:
    """
    Cấu hình tổng hợp toàn hệ thống Driver Guardian GCN.
    Tất cả các siêu tham số được quản lý tập trung và phân nhóm rõ ràng.
    """
    # =====================================================================
    # 1. CẤU HÌNH ĐỒ THỊ VÀ TRÍCH XUẤT ĐỘNG HỌC (GRAPH & KINEMATICS)
    # =====================================================================
    num_nodes: int = 68
    nose_idx: int = 30  # Mốc tâm chóp mũi (gốc tọa độ cục bộ)
    left_eye_idx: int = 36  # Khóe mắt ngoài trái (đo IOD)
    right_eye_idx: int = 45  # Khóe mắt ngoài phải (đo IOD)
    eps: float = 1e-6  # Tránh chia cho 0 khi tính khoảng cách IOD
    adaptive_init_scale: float = 1e-5  # Hệ số khởi tạo ma trận kề thích ứng

    # Cặp chỉ số đo độ mở mắt (EAR) & miệng (MAR)
    vertical_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 41), (38, 40), (43, 47), (44, 46),  # Mí mắt trên - dưới
        (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)  # Môi trên - dưới
    ])

    # Cặp chỉ số tiên nghiệm liên vùng (Mắt <-> Miệng, Mũi <-> Cằm)
    cross_region_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 48), (41, 48), (38, 62), (40, 66),  # Mắt trái <-> Miệng
        (44, 54), (46, 54), (43, 62), (47, 66),  # Mắt phải <-> Miệng
        (30, 8), (30, 36), (30, 45), (27, 8)  # Chóp mũi <-> Cằm & Đuôi mắt
    ])

    # =====================================================================
    # 2. CẤU HÌNH MÔ HÌNH (MODEL ARCHITECTURE)
    # =====================================================================
    model_name: str = "stgcn"  # "stgcn" | "transformer"
    in_channels: int = 6  # [x, y, dx, dy, d2x, d2y]
    num_classes: int = 2  # Mặc định 2 lớp (Alert vs Drowsy)

    # Siêu tham số riêng cho Model 1: DriverSTGCN
    stgcn_hidden_dims: List[int] = field(default_factory=lambda: [64, 128, 256])
    stgcn_strides: List[int] = field(default_factory=lambda: [1, 2, 2])
    temporal_kernel: int = 5  # Độ dài kernel tích chập thời gian
    gat_heads: int = 4  # Số attention heads của DenseGATConv
    dropout: float = 0.25  # Tỷ lệ Dropout cho STGCN
    classifier_hidden_dim: Optional[int] = None  # None: mặc định curr_dim // 2

    # Siêu tham số riêng cho Model 2: DriverSTGCNTransformer
    d_model: int = 128  # Kích thước biểu diễn của Transformer
    nhead: int = 4  # Số Attention Heads trong Transformer
    num_transformer_layers: int = 2  # Số lớp TransformerEncoder
    max_len: int = 300  # Chiều dài chuỗi thời gian tối đa
    transformer_dropout: float = 0.2
    stem_channels: List[int] = field(default_factory=lambda: [64, 128])
    stem_strides: List[int] = field(default_factory=lambda: [1, 2])
    dim_feedforward: Optional[int] = None  # None: mặc định d_model * 2
    init_std: float = 0.02  # Độ lệch chuẩn khởi tạo CLS token & PosEmbedding

    # =====================================================================
    # 3. CẤU HÌNH BỘ DỮ LIỆU, INSIGHTFACE & CACHE (DATASET, INSIGHTFACE & CACHE)
    # =====================================================================
    dataset_dir: str = r"/kaggle/input/data-processed-v3/data_processed"  # Thư mục chứa video & manifest
    manifest_filename: str = "dataset_merged_split.csv"
    time_interval: float = 0.1  # 0.05s lấy mẫu thời gian thực (20 FPS)
    window_sec: Optional[float] = None  # Cửa sổ thời gian (giây), None: toàn bộ video
    landmark_model_path: Optional[str] = None  # None: tự tìm hoặc nạp mặc định của InsightFace
    landmark_model_name: str = "buffalo_l"  # Model InsightFace 68 3D landmarks
    det_size: Tuple[int, int] = (320, 320)  # Kích thước đầu vào SCRFD (320x320 giảm 75% anchors so với 640x640)
    class_names: Dict[int, str] = field(default_factory=lambda: {
        0: "Alert",
        1: "Drowsy"
    })
    letterbox_size: Tuple[int, int] = (640, 480)  # Kích thước (w, h) của letterbox đồng nhất cho mọi video; (0, 0) để tắt

    # Cấu hình trích xuất & nạp cache landmark (.pt) - dùng chung cho extract_cache_v2.ipynb và train_v2.ipynb
    use_cache: bool = True  # Bật nạp chuỗi landmark từ file {cache_dir}/{split}.pt trong GCNDataset
    cache_dir: str = "GCN/cache_landmarks"  # Thư mục chứa train.pt, val.pt, test.pt
    cache_splits: List[str] = field(default_factory=lambda: ["train", "val", "test"])
    landmark_batch_size: int = 64  # Batch size suy luận 1k3d68.onnx (Batched GPU Inference)
    extract_num_workers: int = 4  # Số luồng CPU giải mã video song song khi tạo cache
    extract_max_side: int = 0  # 0: tắt thu nhỏ thủ công vì đã có letterbox_size (w, h)
    cache_save_every: int = 100  # Lưu tạm (atomic save) file .pt sau mỗi N video
    resume_cache: bool = True  # Tiếp tục từ các video đã có trong file .pt

    # =====================================================================
    # 4. CẤU HÌNH DATALOADER & HUẤN LUYỆN (DATALOADER & TRAINING)
    # =====================================================================
    batch_size: int = 64
    num_workers: int = 0
    pin_memory: bool = True
    shuffle_train: bool = True

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 50
    warmup_epochs: int = 2
    min_lr_ratio: float = 0.05
    max_grad_norm: float = 1.0
    amp: bool = True
    save_dir: str = "GCN/checkpoints"
    log_dir: Optional[str] = None
    resume: Optional[str] = None
    seed: int = 42

    def __post_init__(self):
        """Tự động điều chỉnh và đồng bộ các thuộc tính phụ thuộc."""
        # Chuẩn hóa letterbox_size về tuple (w, h)
        if isinstance(self.letterbox_size, int):
            self.letterbox_size = (self.letterbox_size, self.letterbox_size) if self.letterbox_size > 0 else (0, 0)
        elif isinstance(self.letterbox_size, (list, tuple)) and len(self.letterbox_size) == 2:
            w, h = int(self.letterbox_size[0]), int(self.letterbox_size[1])
            self.letterbox_size = (w, h) if (w > 0 and h > 0) else (0, 0)
        else:
            self.letterbox_size = (0, 0)

        if isinstance(self.det_size, (list, tuple)) and len(self.det_size) == 2:
            self.det_size = (int(self.det_size[0]), int(self.det_size[1]))

        if self.class_names is None or len(self.class_names) != self.num_classes:
            if self.num_classes == 2:
                self.class_names = {0: "Alert", 1: "Drowsy"}
            elif self.num_classes == 3:
                self.class_names = {0: "Alert", 1: "Fatigue", 2: "Drowsy"}
            else:
                self.class_names = {i: f"Class_{i}" for i in range(self.num_classes)}

    def to_dict(self) -> Dict[str, Any]:
        """Chuyển đổi toàn bộ cấu hình sang dictionary."""
        return asdict(self)

    def save(self, path: str | Path) -> None:
        """Lưu cấu hình sang tệp JSON hoặc YAML (nếu có PyYAML)."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        data = self.to_dict()

        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "w", encoding="utf-8") as f:
                    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
                return
            except ImportError:
                pass  # Fallback to json if PyYAML is not installed

        with open(p, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: str | Path) -> "TrainConfig":
        """Nạp cấu hình từ tệp JSON hoặc YAML."""
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(f"Không tìm thấy file config tại: {p}")

        content = None
        if p.suffix.lower() in (".yaml", ".yml"):
            try:
                import yaml
                with open(p, "r", encoding="utf-8") as f:
                    content = yaml.safe_load(f)
            except ImportError:
                pass

        if content is None:
            with open(p, "r", encoding="utf-8") as f:
                content = json.load(f)

        # Chuyển class_names key sang int nếu nạp từ json (json lưu key dạng string)
        if "class_names" in content and isinstance(content["class_names"], dict):
            content["class_names"] = {int(k): v for k, v in content["class_names"].items()}

        return cls(**content)

    # -----------------------------------------------------------------
    # Helper methods trích xuất kwargs để truyền vào các components
    # -----------------------------------------------------------------
    def get_stgcn_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCN."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "hidden_dims": list(self.stgcn_hidden_dims),
            "strides": list(self.stgcn_strides),
            "dropout": self.dropout,
            "temporal_kernel": self.temporal_kernel,
            "gat_heads": self.gat_heads,
            "classifier_hidden_dim": self.classifier_hidden_dim,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
        }

    def get_transformer_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho mô hình DriverSTGCNTransformer."""
        return {
            "in_channels": self.in_channels,
            "num_classes": self.num_classes,
            "num_nodes": self.num_nodes,
            "d_model": self.d_model,
            "nhead": self.nhead,
            "num_transformer_layers": self.num_transformer_layers,
            "max_len": self.max_len,
            "dropout": self.transformer_dropout,
            "stem_channels": list(self.stem_channels),
            "stem_strides": list(self.stem_strides),
            "dim_feedforward": self.dim_feedforward,
            "init_std": self.init_std,
            "nose_idx": self.nose_idx,
            "left_eye_idx": self.left_eye_idx,
            "right_eye_idx": self.right_eye_idx,
            "eps": self.eps,
        }

    def get_extractor_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho FaceLandmarkExtractor."""
        return {
            "model_path": self.landmark_model_path,
            "time_interval": self.time_interval,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "landmark_batch_size": self.landmark_batch_size,
            "letterbox_size": self.letterbox_size,
        }

    def get_dataset_kwargs(self, split: str = "train") -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho GCNDataset."""
        return {
            "dataset_dir": self.dataset_dir,
            "split": split,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "use_cache": self.use_cache,
            "cache_dir": self.cache_dir,
            "landmark_batch_size": self.landmark_batch_size,
        }

    def get_dataloader_kwargs(self) -> Dict[str, Any]:
        """Trích xuất tham số thuần túy cho build_gcn_dataloaders."""
        return {
            "dataset_dir": self.dataset_dir,
            "batch_size": self.batch_size,
            "num_workers": self.num_workers,
            "time_interval": self.time_interval,
            "window_sec": self.window_sec,
            "manifest_filename": self.manifest_filename,
            "pin_memory": self.pin_memory,
            "shuffle_train": self.shuffle_train,
            "landmark_model_path": self.landmark_model_path,
            "letterbox_size": self.letterbox_size,
            "device": self.device,
            "det_size": self.det_size,
            "model_name": self.landmark_model_name,
            "use_cache": self.use_cache,
            "cache_dir": self.cache_dir,
            "landmark_batch_size": self.landmark_batch_size,
        }


DEFAULT_CONFIG = TrainConfig()


def get_default_config(num_classes: int = 2) -> TrainConfig:
    """Tạo đối tượng TrainConfig với số nhãn phân loại tùy chọn (2 hoặc 3 lớp)."""
    return TrainConfig(num_classes=num_classes)


def save_config(config: TrainConfig, path: str | Path) -> None:
    """Lưu đối tượng TrainConfig ra tệp JSON hoặc YAML."""
    config.save(path)


def load_config(path: str | Path) -> TrainConfig:
    """Nạp tệp cấu hình từ JSON hoặc YAML và trả về TrainConfig."""
    return TrainConfig.load(path)

In [5]:
# Cell 4: Bộ trích xuất 68 điểm mốc InsightFace tốc độ cao (Batched GPU Inference) và Dataset giải mã video đa luồng
# =====================================================================
# 1. ĐỌC VIDEO & BỘ TRÍCH XUẤT 68 ĐIỂM MỐC (INSIGHTFACE CUDA, BATCHED)
# =====================================================================
def _parse_letterbox_size(new_size: Optional[int | Tuple[int, int] | List[int]]) -> Tuple[int, int]:
    """Chuẩn hóa tham số kích thước letterbox về bộ (target_w, target_h). Trả về (0, 0) nếu tắt."""
    if new_size is None:
        return (0, 0)
    if isinstance(new_size, int):
        return (new_size, new_size) if new_size > 0 else (0, 0)
    if isinstance(new_size, (tuple, list)) and len(new_size) == 2:
        w, h = int(new_size[0]), int(new_size[1])
        return (w, h) if (w > 0 and h > 0) else (0, 0)
    raise ValueError(f"new_size phải là int hoặc tuple/list (w, h), nhận được: {new_size}")


def letterbox(
        image: np.ndarray,
        new_size: int | Tuple[int, int] = (640, 480),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """
    Resize ảnh giữ nguyên tỷ lệ khung hình gốc với đệm màu viền về kích thước đích (w, h).
    - image: Ảnh BGR numpy array shape (H, W, 3).
    - new_size: Kích thước đích theo (w, h) (vd: (640, 480)) hoặc số nguyên S tương ứng (S, S).
    - color: Màu đệm viền BGR, mặc định (114, 114, 114).
    Trả về ảnh numpy array có shape (target_h, target_w, 3).
    """
    target_w, target_h = _parse_letterbox_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def read_video_frames(
        video_path: str | Path,
        time_interval: float = 0.1,
        window_sec: Optional[float] = None,
        letterbox_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
        max_side: int = 0,
) -> Tuple[List[np.ndarray], float, int, float]:
    """
    Đọc frame từ video chỉ với MỘT lần mở cv2.VideoCapture.
    - time_interval > 0: lấy mẫu cách đều theo thời gian thực; == 0: đọc tuần tự mọi frame.
    - window_sec: None = toàn bộ video; số giây = chỉ lấy cửa sổ đầu tiên (bắt đầu từ 0s).
    - letterbox_size: (w, h) kích thước chuẩn hóa letterbox đồng nhất tỷ lệ cho mọi video.
    - max_side: tùy chọn tương thích ngược (nếu letterbox_size=(0, 0)).
    Trả về (frames, fps, total_frames, duration_sec).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở video: {video_path}")

    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
        if not math.isfinite(fps) or fps <= 0:
            fps = 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            raise IOError(f"Video không có frame hợp lệ: {video_path}")
        duration_sec = total_frames / fps
        win = duration_sec if window_sec is None else min(float(window_sec), duration_sec)

        if time_interval == 0:
            num = total_frames if window_sec is None else max(1, min(int(round(win * fps)), total_frames))
            indices = np.arange(num, dtype=int)
        else:
            timestamps = np.arange(0, win, time_interval)
            if len(timestamps) == 0:
                timestamps = np.array([0.0])
            indices = np.clip(np.rint(timestamps * fps).astype(int), 0, total_frames - 1)

        target_w, target_h = _parse_letterbox_size(letterbox_size)
        frames: List[np.ndarray] = []
        curr_frame = 0
        for target_idx in indices:
            if target_idx < curr_frame and frames:
                frames.append(frames[-1])  # Chỉ số trùng lặp (time_interval < 1/fps)
                continue
            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1
            ret, frame = cap.read()
            if not ret:
                break
            if target_w > 0 and target_h > 0:
                frame = letterbox(frame, new_size=(target_w, target_h))
            elif max_side and max(frame.shape[:2]) > max_side:
                scale = max_side / float(max(frame.shape[:2]))
                frame = cv2.resize(
                    frame,
                    (max(1, round(frame.shape[1] * scale)), max(1, round(frame.shape[0] * scale))),
                    interpolation=cv2.INTER_AREA,
                )
            frames.append(frame)
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được frame nào từ video: {video_path}")
    # Bù frame cuối nếu video kết thúc sớm hơn dự kiến
    if len(frames) < len(indices):
        frames.extend([frames[-1]] * (len(indices) - len(frames)))
    return frames, fps, total_frames, duration_sec


class FaceLandmarkExtractor:
    """
    Trích xuất 68 điểm mốc 3D (lấy x, y) bằng InsightFace, tối ưu tốc độ:
      - Chỉ nạp det_*.onnx (SCRFD) + 1k3d68.onnx, không nạp recognition / genderage / 2d106det.
      - max_num=1 (metric="max"): chỉ giữ khuôn mặt lớn nhất trước khi chạy 1k3d68.
      - Chạy 1k3d68.onnx theo lô (batched) cho toàn bộ frame hợp lệ của video.
      - Chỉ lấy 68 điểm cuối, biến đổi ngược Affine vector hóa bằng np.einsum, bỏ qua 3D Pose (lstsq).
    Tọa độ chuẩn hóa theo kích thước letterbox (w, h) của frame, kết quả trong [0, 1].
    """

    def __init__(
            self,
            model_path: Optional[str | Path] = None,
            time_interval: float = 0.0,
            device: str = "cuda",
            det_size: Tuple[int, int] = (320, 320),
            model_name: str = "buffalo_l",
            landmark_batch_size: int = 64,
            det_thresh: float = 0.5,
            letterbox_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
    ):
        self.model_path = Path(model_path) if model_path is not None else None
        self.time_interval = float(time_interval)
        self.device = device.lower().strip()
        self.det_size = tuple(det_size)
        self.model_name = model_name
        self.landmark_batch_size = max(1, int(landmark_batch_size))
        self.det_thresh = det_thresh
        self.letterbox_size = _parse_letterbox_size(letterbox_size)
        self.det = None
        self.lmk = None

    def _init_landmarker(self):
        """Khởi tạo trễ (lazy init) chỉ 2 mô hình ONNX cần thiết."""
        import glob
        import onnxruntime as ort
        from insightface.model_zoo import model_zoo
        from insightface.utils import ensure_available

        ort.set_default_logger_severity(3)  # Tắt cảnh báo shape khi chạy dynamic batch

        available_providers = ort.get_available_providers()
        use_cuda = self.device.startswith("cuda") and ("CUDAExecutionProvider" in available_providers)

        if self.device.startswith("cuda") and not use_cuda:
            logger.warning(
                f"[WARNING] Không tìm thấy CUDAExecutionProvider (khả dụng: {available_providers})! "
                "Mô hình InsightFace sẽ chạy trên CPU."
            )

        if use_cuda:
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
            ctx_id = 0
        else:
            providers = ["CPUExecutionProvider"]
            ctx_id = -1

        root = str(self.model_path) if (self.model_path is not None and self.model_path.is_dir()) else "~/.insightface"
        model_dir = ensure_available("models", self.model_name, root=root)

        det_files = sorted(glob.glob(os.path.join(model_dir, "det_*.onnx")))
        lmk_file = os.path.join(model_dir, "1k3d68.onnx")
        if not det_files or not os.path.isfile(lmk_file):
            raise FileNotFoundError(
                f"Thiếu det_*.onnx hoặc 1k3d68.onnx trong {model_dir} (model_name='{self.model_name}')."
            )

        self.det = model_zoo.get_model(det_files[0], providers=providers)
        self.lmk = model_zoo.get_model(lmk_file, providers=providers)
        self.det.prepare(ctx_id, input_size=self.det_size, det_thresh=self.det_thresh)
        self.lmk.prepare(ctx_id)
        if hasattr(self.lmk, "require_pose"):
            self.lmk.require_pose = False  # Bỏ qua ước lượng 3D Pose (np.linalg.lstsq) không dùng đến
        logger.info(
            f"InsightFace ({self.model_name}) khởi tạo thành công trên provider: {providers[0]} "
            f"(det={os.path.basename(det_files[0])}, det_size={self.det_size}, landmark_batch={self.landmark_batch_size}, letterbox={self.letterbox_size})."
        )

    def release(self) -> None:
        """Giải phóng session ONNX và bộ nhớ GPU."""
        import gc
        self.det = None
        self.lmk = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def extract(self, frames: List[np.ndarray]) -> np.ndarray:
        """
        Nhận danh sách frames BGR và trích xuất chuỗi 68 điểm mốc chuẩn hóa trong [0, 1].
        Trả về numpy array kích thước: (T, 68, 2)
        """
        if self.det is None or self.lmk is None:
            self._init_landmarker()

        from insightface.utils import face_align

        T = len(frames)
        if T == 0:
            return np.zeros((0, 68, 2), dtype=np.float32)

        # Nếu letterbox đang bật và có frame chưa đúng kích thước (w, h), letterbox các frame đó
        target_w, target_h = self.letterbox_size
        if target_w > 0 and target_h > 0:
            processed_frames = []
            for f in frames:
                if f.shape[1] != target_w or f.shape[0] != target_h:
                    processed_frames.append(letterbox(f, new_size=(target_w, target_h)))
                else:
                    processed_frames.append(f)
            frames = processed_frames

        # 1. Phát hiện khuôn mặt lớn nhất trên từng frame
        bboxes: List[Optional[np.ndarray]] = []
        for frame in frames:
            det, _ = self.det.detect(frame, max_num=1, metric="max")
            bboxes.append(det[0, :4] if (det is not None and det.shape[0] > 0) else None)

        valid_idx = [i for i, b in enumerate(bboxes) if b is not None]
        lm68 = np.zeros((T, 68, 2), dtype=np.float32)
        found = np.zeros(T, dtype=bool)

        # 2. Chạy 1k3d68.onnx theo lô trên các frame có khuôn mặt
        isz = int(self.lmk.input_size[0])
        mean = float(self.lmk.input_mean)
        std = float(self.lmk.input_std)
        for s in range(0, len(valid_idx), self.landmark_batch_size):
            chunk = valid_idx[s:s + self.landmark_batch_size]
            aimgs, ims = [], []
            for i in chunk:
                bbox = bboxes[i]
                w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
                center = (bbox[2] + bbox[0]) / 2, (bbox[3] + bbox[1]) / 2
                scale = isz / (max(w, h) * 1.5)
                aimg, M = face_align.transform(frames[i], center, isz, scale, 0)
                aimgs.append(aimg)
                ims.append(cv2.invertAffineTransform(M))

            blob = cv2.dnn.blobFromImages(aimgs, 1.0 / std, (isz, isz), (mean, mean, mean), swapRB=True)
            pred = self.lmk.session.run(self.lmk.output_names, {self.lmk.input_name: blob})[0]
            pred = pred.reshape(len(chunk), -1, 3)[:, -68:, :2].astype(np.float32)  # chỉ 68 điểm cuối, (x, y)
            pred = (pred + 1.0) * (isz // 2)

            homo = np.concatenate([pred, np.ones((len(chunk), 68, 1), dtype=np.float32)], axis=-1)  # (n, 68, 3)
            im_batch = np.stack(ims).astype(np.float32)  # (n, 2, 3)
            out = np.einsum("nkj,nij->nki", homo, im_batch)  # (n, 68, 2)

            for j, i in enumerate(chunk):
                fh, fw = frames[i].shape[:2]
                lm68[i, :, 0] = out[j, :, 0] / float(max(1, fw))
                lm68[i, :, 1] = out[j, :, 1] / float(max(1, fh))
                found[i] = True

        # 3. Điền các frame mất mặt: forward-fill, sau đó backward-fill các frame đầu
        if found.any():
            last = None
            for i in range(T):
                if found[i]:
                    last = lm68[i]
                elif last is not None:
                    lm68[i] = last
            first = int(np.argmax(found))
            if first > 0:
                lm68[:first] = lm68[first]

        return lm68

    def extract_from_video(
            self,
            video_path: str | Path,
            time_interval: Optional[float] = None,
            window_sec: Optional[float] = None,
            letterbox_size: Optional[int | Tuple[int, int] | List[int]] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], Dict[str, Any]]:
        """
        Đọc frame trực tiếp từ video file và trích xuất chuỗi 68 landmark.
        - time_interval: khoảng thời gian (giây) giữa 2 frame lấy mẫu (0: đọc mọi frame, None: dùng self.time_interval).
        - window_sec: thời lượng cửa sổ (giây) cần lấy. None = toàn bộ video.
        - letterbox_size: (w, h) kích thước chuẩn hóa letterbox (None = dùng self.letterbox_size).
        """
        ti = self.time_interval if time_interval is None else float(time_interval)
        if ti < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {ti}")

        lbs = self.letterbox_size if letterbox_size is None else _parse_letterbox_size(letterbox_size)
        frames, fps, total_frames, duration_sec = read_video_frames(
            video_path, ti, window_sec, letterbox_size=lbs
        )
        lm68 = self.extract(frames)
        meta = {
            "video_path": str(video_path),
            "fps": fps,
            "total_frames": total_frames,
            "sampled_frames": len(frames),
            "duration_sec": duration_sec,
            "time_interval": ti,
            "letterbox_size": lbs,
        }
        return lm68, frames, meta


# =====================================================================
# 2. DANH SÁCH MẪU TỪ MANIFEST & DATASET GIẢI MÃ VIDEO ĐA LUỒNG
# =====================================================================
def load_manifest_samples(cfg: TrainConfig, split: str) -> List[Dict[str, Any]]:
    """Nạp danh sách video của một split từ manifest CSV (hoặc quét thư mục dự phòng). Khóa 'key' trùng với GCNDataset."""
    root = Path(cfg.dataset_dir)
    manifest = root / cfg.manifest_filename
    samples: List[Dict[str, Any]] = []

    if manifest.is_file():
        with manifest.open(encoding="utf-8-sig", newline="") as f:
            for row in csv.DictReader(f):
                if row.get("split", "").strip().lower() != split:
                    continue
                rel_path = row.get("rel_path") or row.get("processed_path")
                if not rel_path:
                    continue
                rel_posix = Path(rel_path.replace("\\", "/")).as_posix()
                samples.append({
                    "key": rel_posix,
                    "path": root / rel_posix,
                    "label": int(row["label"]),
                    "video_id": row.get("video_id") or row.get("clip_id") or Path(rel_path).stem,
                })
    else:
        for label, folder in ((0, "0_alert"), (1, "1_drowsy")):
            split_dir = root / split / folder
            if split_dir.is_dir():
                for p in sorted(split_dir.glob("*.mp4")):
                    samples.append({
                        "key": p.relative_to(root).as_posix(),
                        "path": p,
                        "label": label,
                        "video_id": p.stem,
                    })
    return samples


class VideoDecodeDataset(Dataset):
    """
    Dataset mỏng chỉ giải mã video và letterbox (toàn thời lượng, lấy mẫu theo time_interval) để chạy trên các worker CPU
    song song với GPU. Worker KHÔNG khởi tạo CUDA/ONNX nên an toàn với num_workers > 0.
    """

    def __init__(
            self,
            samples: List[Dict[str, Any]],
            time_interval: float,
            letterbox_size: Optional[int | Tuple[int, int] | List[int]] = (640, 480),
            max_side: int = 0,
    ):
        self.samples = samples
        self.time_interval = float(time_interval)
        self.letterbox_size = _parse_letterbox_size(letterbox_size)
        self.max_side = int(max_side)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        sample = self.samples[index]
        try:
            frames, fps, total_frames, duration_sec = read_video_frames(
                sample["path"],
                self.time_interval,
                window_sec=None,
                letterbox_size=self.letterbox_size,
                max_side=self.max_side,
            )
            return {
                "index": index,
                "frames": torch.from_numpy(np.stack(frames)),  # (T, H, W, 3) uint8
                "fps": fps,
                "total_frames": total_frames,
                "duration_sec": duration_sec,
                "error": "",
            }
        except Exception as exc:
            return {"index": index, "frames": None, "fps": 0.0, "total_frames": 0, "duration_sec": 0.0,
                    "error": str(exc)}

In [6]:
# Cell 5: Pipeline trích xuất cache landmark hướng đối tượng (OOP) LandmarkCacheBuilder
# Lưu {cache_dir}/train.pt, val.pt, test.pt; hỗ trợ Auto-Resume, Atomic Save và ngắt an toàn (Ctrl+C / Stop).
# =====================================================================
# LỚP QUẢN LÝ TRÍCH XUẤT CACHE HƯỚNG ĐỐI TƯỢNG (OOP LANDMARK CACHE BUILDER)
# =====================================================================
class LandmarkCacheBuilder:
    """
    Lớp Builder hướng đối tượng quản lý toàn bộ vòng đời tạo cache landmark:
    1. Khởi tạo và đồng bộ siêu tham số cấu hình (TrainConfig dùng chung với train_v2.ipynb).
    2. Giải mã video đa luồng trên CPU (VideoDecodeDataset) và trích xuất InsightFace theo lô trên GPU.
    3. Lưu mỗi split thành một file .pt: {"samples": {key: {"landmarks": (2, T_full, 68), ...}}, ...}.
    4. Auto-Resume (bỏ qua video đã có), Atomic Save định kỳ, bắt KeyboardInterrupt an toàn.
    """

    # -----------------------------------------------------------------
    # 1. CÁC PHƯƠNG THỨC TIỆN ÍCH HỆ THỐNG (STATIC METHODS)
    # -----------------------------------------------------------------
    @staticmethod
    def seed_everything(seed: int = 42) -> None:
        """Cố định seed để đảm bảo khả năng tái lập kết quả."""
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

    @staticmethod
    def configure_logging(log_file: Path | str) -> None:
        """Cấu hình logging đồng thời ra Console và file extract.log."""
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)

        logger.setLevel(logging.INFO)
        logger.propagate = False
        for handler in logger.handlers[:]:
            handler.close()
            logger.removeHandler(handler)

        formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

        file_handler = logging.FileHandler(log_file, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        stream_handler = logging.StreamHandler(sys.stdout)
        stream_handler.setFormatter(formatter)
        logger.addHandler(stream_handler)

    @staticmethod
    def atomic_save(state: Any, path: str | Path) -> None:
        """Lưu tệp an toàn (Atomic Save qua file tạm) để tránh hỏng file .pt khi bị ngắt đột ngột."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
        try:
            with os.fdopen(fd, "wb") as f:
                torch.save(state, f)
                f.flush()
                os.fsync(f.fileno())
            os.replace(tmp_file, path)
        finally:
            Path(tmp_file).unlink(missing_ok=True)

    @staticmethod
    def verify_cache_file(path: str | Path) -> Dict[str, Any]:
        """Kiểm tra tính toàn vẹn của một file cache .pt (cấu trúc, shape (2, T, 68), giá trị hữu hạn)."""
        path = Path(path)
        report: Dict[str, Any] = {"path": str(path), "ok": False, "num_samples": 0, "errors": []}
        if not path.is_file():
            report["errors"].append("Không tìm thấy file")
            return report
        try:
            try:
                payload = torch.load(path, map_location="cpu", weights_only=True)
            except TypeError:
                payload = torch.load(path, map_location="cpu")
        except Exception as exc:
            report["errors"].append(f"Không đọc được file: {exc}")
            return report

        samples = payload.get("samples", {})
        report["num_samples"] = len(samples)
        total_frames = 0
        for key, item in samples.items():
            lm = item.get("landmarks")
            if lm is None or lm.ndim != 3 or lm.shape[0] != 2 or lm.shape[2] != 68 or lm.shape[1] <= 0:
                report["errors"].append(f"{key}: shape landmarks không hợp lệ")
            elif not torch.isfinite(lm).all():
                report["errors"].append(f"{key}: có giá trị NaN/Inf")
            else:
                total_frames += lm.shape[1]
        report["total_frames"] = total_frames
        report["ok"] = len(report["errors"]) == 0
        return report

    # -----------------------------------------------------------------
    # 2. KHỞI TẠO ĐỐI TƯỢNG (INITIALIZATION)
    # -----------------------------------------------------------------
    def __init__(self, cfg: Optional[TrainConfig] = None):
        self.cfg = cfg if cfg is not None else TrainConfig()
        self.cfg.__post_init__()

        self.cache_dir = Path(self.cfg.cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.configure_logging(self.cache_dir / "extract.log")
        self.seed_everything(self.cfg.seed)
        self.cfg.save(self.cache_dir / "config.json")

        self.extractor = FaceLandmarkExtractor(**self.cfg.get_extractor_kwargs())

        logger.info("=" * 70)
        logger.info("KHỞI TẠO PIPELINE TRÍCH XUẤT CACHE LANDMARK (OOP) - EXTRACT CACHE V2")
        logger.info("=" * 70)
        logger.info(f"Dataset root                   : {self.cfg.dataset_dir}")
        logger.info(f"Thư mục cache (.pt)            : {self.cache_dir}")
        logger.info(f"Các split                      : {self.cfg.cache_splits}")
        logger.info(f"Time interval                  : {self.cfg.time_interval}s (toàn thời lượng video)")
        logger.info(f"Letterbox size (w, h)          : {self.cfg.letterbox_size}")
        logger.info(
            f"Mô hình Landmark (InsightFace) : {self.cfg.landmark_model_name} (Det: {self.cfg.det_size}, batch: {self.cfg.landmark_batch_size})")
        logger.info(f"Thiết bị trích xuất Landmark   : {self.cfg.device}")
        logger.info(f"Worker giải mã video (CPU)     : {self.cfg.extract_num_workers}")

    # -----------------------------------------------------------------
    # 3. TRÍCH XUẤT TỪNG SPLIT
    # -----------------------------------------------------------------
    def _new_cache(self, split: str) -> Dict[str, Any]:
        return {
            "format_version": 1,
            "split": split,
            "time_interval": float(self.cfg.time_interval),
            "letterbox_size": tuple(self.cfg.letterbox_size),
            "det_size": tuple(self.cfg.det_size),
            "model_name": self.cfg.landmark_model_name,
            "num_samples": 0,
            "samples": {},
        }

    def _load_existing_cache(self, split: str, path: Path) -> Dict[str, Any]:
        """Nạp file .pt đã có để Auto-Resume; khởi tạo mới nếu chưa có hoặc cấu hình không còn khớp."""
        if not (self.cfg.resume_cache and path.is_file()):
            return self._new_cache(split)
        try:
            try:
                payload = torch.load(path, map_location="cpu", weights_only=True)
            except TypeError:
                payload = torch.load(path, map_location="cpu")
        except Exception as exc:
            logger.warning(f"Không đọc được {path.name} ({exc}). Trích xuất lại từ đầu.")
            return self._new_cache(split)

        if abs(float(payload.get("time_interval", -1.0)) - float(self.cfg.time_interval)) > 1e-9:
            logger.warning(f"{path.name} có time_interval khác cấu hình hiện tại. Trích xuất lại từ đầu.")
            return self._new_cache(split)
        cached_lbs = payload.get("letterbox_size", ())
        if tuple(cached_lbs) != tuple(self.cfg.letterbox_size):
            logger.warning(
                f"{path.name} tạo với letterbox_size={cached_lbs} khác cấu hình hiện tại {self.cfg.letterbox_size}. Trích xuất lại từ đầu."
            )
            return self._new_cache(split)
        if tuple(payload.get("det_size", ())) != tuple(self.cfg.det_size):
            logger.warning(f"{path.name} tạo với det_size khác cấu hình hiện tại; vẫn tiếp tục các mẫu còn thiếu.")
        return payload

    def extract_split(self, split: str) -> Dict[str, Any]:
        """Trích xuất toàn bộ video của một split và lưu {cache_dir}/{split}.pt."""
        path = self.cache_dir / f"{split}.pt"
        samples = load_manifest_samples(self.cfg, split)
        stats: Dict[str, Any] = {
            "split": split, "total": len(samples), "cached_before": 0, "new": 0, "failed": [],
            "frames": 0, "seconds": 0.0, "interrupted": False, "path": str(path),
        }
        if not samples:
            logger.warning(f"Không có mẫu nào trong split '{split}'. Bỏ qua.")
            return stats

        cache = self._load_existing_cache(split, path)
        done = cache["samples"]
        todo = [s for s in samples if s["key"] not in done]
        stats["cached_before"] = len(samples) - len(todo)
        logger.info("=" * 60)
        logger.info(
            f"BẮT ĐẦU TRÍCH XUẤT SPLIT [{split.upper()}]: tổng {len(samples)} | đã có {stats['cached_before']} | cần trích xuất {len(todo)}")
        logger.info("=" * 60)

        if todo:
            nw = max(0, int(self.cfg.extract_num_workers))
            if nw > 0 and os.name == "nt":
                logger.warning(
                    "Windows dùng 'spawn' nên không chia sẻ được lớp định nghĩa trong notebook: dùng extract_num_workers=0.")
                nw = 0
            loader = DataLoader(
                VideoDecodeDataset(todo, self.cfg.time_interval, letterbox_size=self.cfg.letterbox_size),
                batch_size=None, shuffle=False, num_workers=nw,
                prefetch_factor=2 if nw > 0 else None, pin_memory=False,
            )

            t0 = time.time()
            since_save = 0
            try:
                for item in tqdm(loader, total=len(todo), desc=f"Extracting [{split}]", unit="video",
                                 dynamic_ncols=True):
                    sample = todo[int(item["index"])]
                    if item["frames"] is None:
                        logger.error(f"Lỗi đọc video '{sample['path']}': {item['error']}")
                        stats["failed"].append({"key": sample["key"], "error": item["error"]})
                        continue

                    frames = list(item["frames"].numpy())
                    try:
                        lm68 = self.extractor.extract(frames)  # (T, 68, 2)
                    except Exception as exc:
                        logger.error(f"Lỗi trích xuất landmark '{sample['path']}': {exc}")
                        stats["failed"].append({"key": sample["key"], "error": str(exc)})
                        continue

                    done[sample["key"]] = {
                        "landmarks": torch.from_numpy(lm68).permute(2, 0, 1).contiguous(),  # (2, T_full, 68)
                        "label": int(sample["label"]),
                        "video_id": str(sample["video_id"]),
                        "path": str(sample["path"]),
                        "fps": float(item["fps"]),
                        "total_frames": int(item["total_frames"]),
                        "duration_sec": float(item["duration_sec"]),
                    }
                    stats["new"] += 1
                    stats["frames"] += len(frames)
                    since_save += 1
                    if since_save >= self.cfg.cache_save_every:
                        cache["num_samples"] = len(done)
                        self.atomic_save(cache, path)
                        since_save = 0
            except KeyboardInterrupt:
                stats["interrupted"] = True
                logger.warning("Quá trình trích xuất bị ngắt bởi người dùng. Đang lưu các mẫu đã hoàn tất...")
            finally:
                cache["num_samples"] = len(done)
                self.atomic_save(cache, path)
                stats["seconds"] = time.time() - t0
        else:
            logger.info(f"Split '{split}' đã đủ 100% mẫu trong {path.name}. Bỏ qua trích xuất.")

        stats["size_mb"] = path.stat().st_size / (1024 * 1024) if path.is_file() else 0.0
        return stats

    # -----------------------------------------------------------------
    # 4. PHƯƠNG THỨC CHÍNH ĐIỀU PHỐI (FIT)
    # -----------------------------------------------------------------
    def fit(self) -> Dict[str, Any]:
        """Trích xuất toàn bộ các split trong cfg.cache_splits, kiểm tra tính toàn vẹn và in báo cáo tổng kết."""
        summary: Dict[str, Any] = {}
        start = time.time()
        for split in self.cfg.cache_splits:
            stats = self.extract_split(split)
            stats["verify"] = self.verify_cache_file(self.cache_dir / f"{split}.pt")
            summary[split] = stats
            if stats["interrupted"]:
                break

        self.extractor.release()  # Giải phóng VRAM của InsightFace trước khi huấn luyện

        logger.info("=" * 70)
        logger.info(f"HOÀN TẤT TRÍCH XUẤT CACHE sau {(time.time() - start) / 60:.2f} phút.")
        for split, st in summary.items():
            vf = st["verify"]
            speed = (st["new"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            fps_ = (st["frames"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            logger.info(
                f"  [{split:5s}] tổng={st['total']} | mới={st['new']} | lỗi={len(st['failed'])} | "
                f"{speed:.2f} video/s ({fps_:.0f} frame/s) | {st.get('size_mb', 0.0):.1f} MB | "
                f"verify={'OK' if vf['ok'] else 'LỖI'} ({vf['num_samples']} mẫu)"
            )
            for err in vf["errors"][:5]:
                logger.warning(f"      - {err}")
        return summary

In [ ]:
# Cell 6: Khởi tạo cấu hình TrainConfig, đối tượng LandmarkCacheBuilder và bắt đầu trích xuất cache .pt
config = TrainConfig()
builder = LandmarkCacheBuilder(config)
cache_summary = builder.fit()

[2026-11-03 03:23:46][INFO] ======================================================================
[2026-11-03 03:23:46][INFO] KHỞI TẠO PIPELINE TRÍCH XUẤT CACHE LANDMARK (OOP) - EXTRACT CACHE V2
[2026-11-03 03:23:46][INFO] ======================================================================
[2026-11-03 03:23:46][INFO] Dataset root                   : /kaggle/input/data-processed-v3/data_processed
[2026-11-03 03:23:46][INFO] Thư mục cache (.pt)            : GCN/cache_landmarks
[2026-11-03 03:23:46][INFO] Các split                      : ['train', 'val', 'test']
[2026-11-03 03:23:46][INFO] Time interval                  : 0.1s (toàn thời lượng video)
[2026-11-03 03:23:46][INFO] Letterbox size (w, h)          : (640, 480)
[2026-11-03 03:23:46][INFO] Mô hình Landmark (InsightFace) : buffalo_l (Det: (320, 320), batch: 64)
[2026-11-03 03:23:46][INFO] Thiết bị trích xuất Landmark   : cuda
[2026-11-03 03:23:46][INFO] Worker giải mã video (CPU)     : 4
[2026-11-03 03:23:47][INFO] ===========

Extracting [train]:   0%|          | 0/689 [00:00<?, ?video/s]

[2026-11-03 03:23:51][INFO] InsightFace (buffalo_l) khởi tạo thành công trên provider: CUDAExecutionProvider (det=det_10g.onnx, det_size=(320, 320), landmark_batch=64, letterbox=(640, 480)).


Extracting [train]: 100%|██████████| 689/689 [13:56<00:00,  1.21s/video]


[2026-11-03 03:37:48][INFO] ============================================================
[2026-11-03 03:37:48][INFO] BẮT ĐẦU TRÍCH XUẤT SPLIT [VAL]: tổng 1090 | đã có 0 | cần trích xuất 1090
[2026-11-03 03:37:48][INFO] ============================================================


Extracting [val]: 100%|██████████| 1090/1090 [19:31<00:00,  1.08s/video]


[2026-11-03 03:57:20][INFO] ============================================================
[2026-11-03 03:57:20][INFO] BẮT ĐẦU TRÍCH XUẤT SPLIT [TEST]: tổng 1018 | đã có 0 | cần trích xuất 1018
[2026-11-03 03:57:20][INFO] ============================================================


Extracting [test]: 100%|██████████| 1018/1018 [17:56<00:00,  1.06s/video]


[2026-11-03 04:15:17][INFO] ======================================================================
[2026-11-03 04:15:17][INFO] HOÀN TẤT TRÍCH XUẤT CACHE sau 51.52 phút.
[2026-11-03 04:15:17][INFO]   [train] tổng=5206 | mới=689 | lỗi=0 | 0.82 video/s (140 frame/s) | 412.9 MB | verify=OK (5206 mẫu)
[2026-11-03 04:15:17][INFO]   [val  ] tổng=1090 | mới=1090 | lỗi=0 | 0.93 video/s (142 frame/s) | 86.9 MB | verify=OK (1090 mẫu)
[2026-11-03 04:15:17][INFO]   [test ] tổng=1018 | mới=1018 | lỗi=0 | 0.95 video/s (140 frame/s) | 78.6 MB | verify=OK (1018 mẫu)


In [14]:
!zip -r hazards_results.zip GCN/cache_landmarks

  adding: GCN/cache_landmarks/ (stored 0%)
  adding: GCN/cache_landmarks/extract.log (deflated 82%)
  adding: GCN/cache_landmarks/test.pt (deflated 15%)
  adding: GCN/cache_landmarks/val.pt (deflated 15%)
  adding: GCN/cache_landmarks/config.json (deflated 72%)
  adding: GCN/cache_landmarks/train.pt (deflated 15%)


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
!cp /content/hazards_results.zip /content/drive/MyDrive/